<div dir="rtl" style="text-align: right;">

# دليل استخدام مكتبة **dextra**

مكتبة خفيفة لتحليل البيانات الاستكشافي (EDA) مبنية فوق pandas و seaborn و plotly.
هذا الدفتر يستعرض الدوال الثلاث الأساسية للمكتبة مع شرح مقابل باللغة الإنجليزية.

</div>

---

# `dextra` usage guide

A lightweight exploratory-data-analysis (EDA) toolkit on top of pandas,
seaborn and plotly. This notebook walks through the three public
functions with parallel Arabic/English commentary.

<div dir="rtl" style="text-align: right;">

## 1) التثبيت

قبل البدء، ثبّت المكتبة من المصدر (إذا كان المشروع محلياً) أو من GitHub:

</div>

## 1) Installation

Install from source (local project) or from GitHub before running the
following cells.

In [ ]:
# تثبيت من مصدر محلي / Install from local source
# !pip install -e ..

# أو تثبيت من GitHub / Or install from GitHub
# !pip install git+https://github.com/ahmedabdeltawab/dextra.git

<div dir="rtl" style="text-align: right;">

## 2) الاستيراد وتوليد بيانات نموذجية

سنولّد DataFrame افتراضية تحوي أعمدة لها خصائص إحصائية مختلفة حتى نرى كيف تتعامل المكتبة مع كل نوع.

</div>

## 2) Imports and a sample dataset

We build a synthetic DataFrame with columns that have distinct statistical profiles so every feature is exercised.

In [ ]:
import numpy as np
import pandas as pd

import dextra as dx

print(f"dextra version: {dx.__version__}")

In [ ]:
rng = np.random.default_rng(seed=42)
N = 500

# بيانات تبدو كمبيعات تجزئة — a retail-like synthetic dataset
df = pd.DataFrame({
    "price":       rng.normal(loc=100, scale=15, size=N),           # توزيع طبيعي
    "quantity":    rng.integers(low=1, high=20, size=N),            # عدد صحيح
    "discount_%":  rng.beta(a=2, b=5, size=N) * 40,                 # موزّع بيتا، ملتوٍ
    "rating":      rng.choice([1, 2, 3, 4, 5], size=N, p=[.05,.1,.2,.4,.25]),
    "revenue":     rng.gamma(shape=2, scale=50, size=N),             # غاما، له ذيل أيمن طويل
})

# حقن بعض القيم المفقودة والشاذة / inject some NaNs and outliers
df.loc[rng.integers(0, N, 20), "price"] = np.nan
df.loc[rng.integers(0, N, 5),  "revenue"] *= 10

df.head()

<div dir="rtl" style="text-align: right;">

## 3) `describe_numeric` — ملخّص رقمي غني

بديل أغنى من `df.describe()` يُظهر في جدول واحد 21 مقياساً لكل عمود عددي، مع تنسيق جاهز للعرض:

* مقاييس النزعة المركزية: `mean`, `median`, `modes`.
* مقاييس التشتّت: `std`, `var`, `IQR`, `cv_%`.
* الشكل: `skewness`, `kurtosis`.
* الحدود والقيم الشاذة: `lower_bound`, `upper_bound`, `outliers_count`, `outliers_%`.
* الجودة: `count`, `missing`, `unique`.

</div>

## 3) `describe_numeric` — a rich numeric summary

A drop-in replacement for `df.describe()` that reports 21 metrics per
numeric column in a single, scannable table:

* Central tendency: `mean`, `median`, `modes`
* Dispersion: `std`, `var`, `IQR`, `cv_%`
* Shape: `skewness`, `kurtosis`
* Outlier bounds: `lower_bound`, `upper_bound`, `outliers_count`, `outliers_%`
* Quality: `count`, `missing`, `unique`

In [ ]:
# الاستدعاء الأبسط — أعمدة عددية افتراضياً
# Simplest call — numeric columns, defaults
dx.describe_numeric(df)

In [ ]:
# اختيار أعمدة معيّنة وضبط عدد الخانات العشرية
# Select specific columns and control decimal places
dx.describe_numeric(df, cols=["price", "revenue"], decimals=3)

In [ ]:
# استرجاع النتيجة كـ DataFrame بأرقام خام (للتصدير والحسابات اللاحقة)
# Return a raw (float) DataFrame for further processing / export
summary_raw = dx.describe_numeric(df, return_df=True, raw=True, show=False)
summary_raw.loc[["mean", "std", "IQR", "outliers_count"]]

In [ ]:
# تعديل عتبة تكي لكشف القيم الشاذة (1.5 افتراضي، 3.0 أكثر تسامحاً)
# Tweak Tukey's multiplier (1.5 default, 3.0 more tolerant)
dx.describe_numeric(df, iqr_multiplier=3.0, cols=["revenue"])

<div dir="rtl" style="text-align: right;">

## 4) `plot_histograms` — مدرّج تكراري مع إحصاءات جانبية

شكل Matplotlib يعرض صفاً لكل عمود: مدرّج تكراري + منحنى KDE على اليسار، ولوحة إحصاءات نصية على اليمين. المتوسط يظهر بخط أحمر متقطّع والوسيط بخط أخضر.

</div>

## 4) `plot_histograms` — histogram with adjacent stats

A Matplotlib figure with one row per column: histogram + KDE on the
left, a monospace statistics panel on the right. Mean is a red dashed
vertical line, median is green.

In [ ]:
# مدرّجات افتراضية لكل الأعمدة العددية
# Default histograms for every numeric column
dx.plot_histograms(df, bins=30)

In [ ]:
# تخصيص الألوان وحفظ الشكل إلى ملف
# Custom colours + save to disk
dx.plot_histograms(
    df,
    cols=["price", "revenue"],
    bins=25,
    hist_color="#4C72B0",
    kde_color="#C44E52",
    save=True,
    output_dir="plots",
    filename="price_revenue.png",
)

In [ ]:
# الحصول على الشكل والإحصاءات معاً (بدون عرض)
# Get figure + summary without rendering
fig, hist_summary = dx.plot_histograms(
    df,
    show=False,
    return_fig=True,
    return_df=True,
)
print(type(fig).__name__)
hist_summary.loc[:, ["mean", "std", "iqr", "outliers_count"]]

<div dir="rtl" style="text-align: right;">

## 5) `plot_boxplots` — صناديق Plotly تفاعلية

رسم Plotly أفقي، صف لكل متغير، مع تعليق إحصائي داخل كل صف وخطوط متقطّعة عند حدَّي تكي.

</div>

## 5) `plot_boxplots` — interactive Plotly box-plots

Horizontal Plotly box-plots, one row per column, with an annotated
summary inside each row and dashed lines at Tukey's outlier bounds.

In [ ]:
dx.plot_boxplots(df)

In [ ]:
# تخصيص الألوان حسب العمود
# Per-column colour map
dx.plot_boxplots(
    df,
    cols=["price", "revenue", "discount_%"],
    colors={
        "price":     "#1f77b4",
        "revenue":   "#d62728",
        "discount_%":"#2ca02c",
    },
    template="plotly_white",
    title="Distribution of numerical fields",
)

In [ ]:
# استخدام القالب الداكن وإرجاع الشكل فقط
# Dark template, return the figure object
fig = dx.plot_boxplots(
    df,
    template="plotly_dark",
    show=False,
    return_fig=True,
)
fig

<div dir="rtl" style="text-align: right;">

## 6) الأسماء المختصرة (متوافقة مع الإصدار السابق)

الأسماء القصيرة `numdesc`, `hister`, `boxpl` لا تزال تعمل. كل منها مجرد اسم مستعار يشير إلى الدالة الجديدة.

</div>

## 6) Backwards-compatible short aliases

`numdesc`, `hister`, and `boxpl` still work. Each is just an alias to
the new name.

In [ ]:
assert dx.numdesc is dx.describe_numeric
assert dx.hister is dx.plot_histograms
assert dx.boxpl is dx.plot_boxplots

# المكالمة بالاسم المختصر / short-name call
dx.numdesc(df[["price", "revenue"]], decimals=1)

<div dir="rtl" style="text-align: right;">

## 7) تدفّق عمل EDA مختصر

المكتبة مصمَّمة لتكون أوّل ثلاث خطوات في تحليل أي DataFrame جديد:

1. **`describe_numeric`** — نظرة سريعة على المقاييس (مركز، تشتّت، شكل، جودة).
2. **`plot_histograms`** — فَهم التوزيعات والذيول.
3. **`plot_boxplots`** — تحديد القيم الشاذة بصرياً ومقارنة المتغيرات.

</div>

## 7) A minimal EDA workflow

The library is designed to be the first three steps you run against any
fresh DataFrame:

1. **`describe_numeric`** — a quick read of centre / spread / shape / quality.
2. **`plot_histograms`** — understand distributions and tails.
3. **`plot_boxplots`** — spot outliers visually and compare features.

In [ ]:
# الخطوات الثلاث في تتابع
# All three steps in sequence
dx.describe_numeric(df, decimals=2)
dx.plot_histograms(df, bins=30)
dx.plot_boxplots(df)